# Cheaper model evaluation

## Predict selectively. Audit randomly. Rank defensibly.

**MMLU + SWE-bench Verified · held-out replay · August 2026**

> The central result: prediction can concentrate evaluation effort, while a randomized audit protects the benchmark score when prediction is wrong.

---

# The 30-second version

- We built a prediction-assisted path for evaluating only a subset of benchmark tasks.
- A historical response model predicts the unobserved tasks.
- Randomized sampling with logged probabilities corrects prediction error.
- Models share selected tasks, enabling lower-noise paired comparisons.
- On held-out MMLU models, **5% of tasks produced median Kendall τ = 0.916**.
- On later SWE-bench systems, a sequential design improved the 5% result from **0.617 to 0.656**.

### Bottom line

The evaluation architecture works. Better SWE-specific prediction is the next multiplier.

---

# The problem

A full evaluation spends equally on:

- highly redundant tasks;
- tasks that do not affect the ranking decision;
- model pairs that are already clearly separated;
- and uncertain comparisons that actually need more evidence.

## Our question

**Can we buy fewer observations without turning a prediction model into the ground truth?**

---

# The architecture

![Prediction-assisted evaluation workflow](docs/figures/evaluation-workflow.svg)

### Replaceable prediction layer

The predictor can later be replaced with embeddings, nearest neighbors, matrix completion, or an ensemble. The correction layer remains the same.

---

# What the predictor actually learns

The historical data is a large pass/fail table:

| Historical system | Task 1 | Task 2 | Task 3 | … |
|---|---:|---:|---:|---:|
| System A | pass | fail | pass | … |
| System B | pass | pass | fail | … |
| System C | fail | fail | pass | … |

A low-rank factorization compresses recurring patterns in this table:

- tasks that tend to be solved by the same systems receive similar representations;
- systems with similar success patterns receive similar profiles;
- a new system's first few answers locate it among those historical profiles;
- that profile predicts its probability of passing every unobserved task.

## Important distinction

The prediction chooses where to spend evaluation effort. It is **not accepted as the final score without checking it**.

---

# A concrete 25-task SWE evaluation

Suppose the full benchmark has 500 tasks, but we can afford approximately 25:

| Round | Approximate tasks | What happens |
|---|---:|---|
| **1 · Sentinels** | 5 | Spread across repositories to estimate the new system's response profile |
| **2 · Adaptive training** | 2–3 | Choose tasks where another answer is most useful for updating that profile |
| **3 · Randomized audit** | 17–18 | Sample the remaining tasks with known probabilities and measure prediction error |

The final round deliberately receives most of the budget. It is what lets us correct mistakes made in rounds 1 and 2.

### Why not make every task adaptive?

An adaptive policy over-selects unusual, difficult, or uncertain tasks. Its raw pass rate therefore no longer represents the original 500-task benchmark. The randomized audit restores that connection.

---

# Why the score remains defensible

For task outcome $y_i$, prediction $q_i$, and logged inclusion probability $\pi_i$:

$$
\widehat{\mu}
= \frac{1}{N}\sum_i q_i
+ \frac{1}{N}\sum_{i\in S}\frac{y_i-q_i}{\pi_i}
$$

The first term predicts the full benchmark. The second term audits and corrects that prediction.

### Read it in plain English

1. Start with the predicted average across all tasks.
2. On audited tasks, compare the real outcome with the prediction.
3. Weight each error by how likely that task was to be sampled.
4. Add the weighted average error back to the predicted score.

For example, if the predicted benchmark score is 40% and the weighted audit says predictions were 4 points too optimistic, the reported score becomes 36%.

## The key property

**A weak surrogate increases variance; it does not redefine the benchmark score.**

---

# Evidence design

| | MMLU | SWE-bench Verified |
|---|---:|---:|
| Historical rows used for training | 296 models | 100 systems |
| Held-out evaluation rows | 99 models | 34 later systems |
| Tasks | 14,042 | 500 |
| Content strata | 57 subjects | 12 repositories |
| Split | Entire creator groups held out | Chronologically latest quarter held out |
| Replicates | 10 seeds | 10 seeds |

Outcomes remained hidden until an item was selected. Every candidate model used the same realized item set for paired comparison.

---

# Where the ranking data came from

## MMLU response matrix

Per-item correctness came from the Open LLM Leaderboard artifact `tutorials/data/lb.pickle` in [tinyBenchmarks](https://github.com/felipemaiapolo/tinyBenchmarks), pinned at commit [`e9a8b10`](https://github.com/felipemaiapolo/tinyBenchmarks/commit/e9a8b1031b0340571beb6c9ca3a27891be09a8fd). We transformed it into a dense matrix of **395 models × 14,042 MMLU items**.

## SWE-bench Verified response matrix

Per-instance resolution outcomes came from `evaluation/verified/*/results/results.json` in the official [SWE-bench experiments repository](https://github.com/SWE-bench/experiments), pinned at commit [`1faa91c`](https://github.com/SWE-bench/experiments/commit/1faa91cade0562ba62b66c1c99e71f7b72d96f13). We transformed 134 submissions into a dense **134 systems × 500 issues** matrix.

The checked-in [MMLU manifest](data/processed/mmlu_openllm/manifest.json) and [SWE-bench manifest](data/processed/swebench_verified/manifest.json) record source paths, commits, and SHA-256 checksums. SWE rows represent complete model-plus-agent systems.

---

# Result 1 · Strong MMLU ranking at low cost

![MMLU budget versus ranking fidelity](docs/figures/mmlu-cost-curve.svg)

| Budget | Median tasks/model | Median Kendall τ | Median score MAE | Median interval coverage |
|---:|---:|---:|---:|---:|
| **1%** | 144 | **0.8085** | 0.0296 | 95.5% |
| **5%** | 707 | **0.9157** | 0.0119 | 96.0% |
| **10%** | 1,397 | **0.9404** | 0.0083 | 95.5% |

## The presentation headline

**Using 5% of MMLU, the method recovered the full ranking with τ ≈ 0.92 while maintaining approximately nominal score coverage.**

---

# Result 2 · Sequential evaluation helped SWE at 5%

![SWE-bench budget versus ranking fidelity](docs/figures/swe-sequential-cost-curve.svg)

The statistically safe sequential design reserves its final round for a randomized audit.

| SWE budget | One-shot τ | Sequential τ | Outcome |
|---:|---:|---:|---|
| 2% · about 9 tasks | 0.5268 | **0.5489** | Sequential improvement |
| **5% · about 24 tasks** | 0.6169 | **0.6559** | **Largest useful gain** |
| 10% · about 50 tasks | **0.6955** | 0.6691 | One-shot remains preferable |

At 5%, sequential evaluation gained **0.039 Kendall τ** while retaining median interval coverage of approximately **95.6%**.

---

# What specifically worked

### 1. Historical responses contained reusable structure
Enough signal transferred to rank completely held-out models and later systems.

### 2. Randomized correction protected the estimand
We did not need to assume that the surrogate was calibrated or structurally correct.

### 3. Shared tasks strengthened comparisons
Paired gaps use the same issue outcomes instead of comparing two unrelated samples.

### 4. Sequential learning can help in the lowest-budget regime
On SWE-bench, the intermediate update improved ordering when only about 24 tasks were available.

### 5. The implementation is lightweight
The full replay completed in under 20 seconds; **53 tests**, Ruff, and strict mypy pass.

---

# Correct ranking means allowing ties

A cheap evaluation should not force a total order when the evidence cannot resolve one.

## Decision rule

Declare model A above model B only when the lower confidence bound for their paired gap exceeds the practical threshold $\epsilon$.

Otherwise:

- keep the pair in the same unresolved tier; or
- buy another shared batch targeted at that comparison.

> The display order is convenient. The confidence-backed partial order is the claim.

---

# From corrected scores to a ranking

Every candidate system is evaluated on the **same selected tasks**. For each pair A and B, we calculate:

```text
paired gap on a task = outcome for A − outcome for B
```

We then apply the same prediction-and-audit correction to those task-level gaps. Shared tasks remove much of the task difficulty noise: a hard task affects both systems rather than only one.

The output has two layers:

1. **Display order:** sort systems by their corrected average score.
2. **Supported order:** separate A from B only when the uncertainty interval around their paired gap clears the practical threshold.

If the data cannot separate two systems, they remain in the same tier. That uncertainty provides the decision rule for whether another shared batch is worth buying.

---

# Where SWE-bench needs richer signals

- Only 100 historical systems were available to predict 34 later systems.
- The 500 issues span 12 repositories with different code and failure modes.
- Rows combine the model, agent scaffold, tools, and inference policy.
- The chronological holdout creates real distribution shift.

## Productive interpretation

The randomized correction is doing its job. The next bottleneck is the **SWE-specific surrogate**, not the validity architecture.

---

# The next efficiency gain

Upgrade the surrogate with information the current SVD cannot see:

1. Repository and task-difficulty effects.
2. Issue-text and codebase embeddings.
3. Model-family and agent-scaffold features.
4. Pairwise acquisition focused on unresolved leaderboard neighbors.
5. Fixed-size stratified audit sampling to reduce propensity-weight variance.

### What stays unchanged

Randomized auditing, logged probabilities, paired gaps, and unresolved confidence tiers.

---

# Questions you are likely to get

<details><summary><b>What if the surrogate is wrong?</b></summary><p>The randomized audit estimates and corrects its residual error. A poor surrogate costs efficiency, not a different target score.</p></details>

<details><summary><b>Is 5% of SWE-bench enough for a definitive leaderboard?</b></summary><p>Not yet. It supports useful coarse ordering, but close pairs remain unresolved and should receive more shared tasks.</p></details>

<details><summary><b>Why did sequential lose at 10%?</b></summary><p>The extra low-rank fitting observations did not repay the randomized audit budget they consumed. That motivates a richer repository- and scaffold-aware predictor.</p></details>

<details><summary><b>Are SWE rows rankings of base models?</b></summary><p>No. They rank model-plus-agent systems under the submitted scaffold and evaluation conditions.</p></details>

---

# Closing

## What we demonstrated

A prediction-assisted evaluation system can:

- concentrate evaluation effort using historical response structure;
- correct prediction errors through randomized auditing;
- estimate shared-item model gaps;
- and recover useful rankings from a fraction of the benchmark.

# Predict selectively. Audit randomly. Rank defensibly.

---

# Appendix · Evidence and reproduction

- HTML evidence report: [`REPORT.html`](REPORT.html)
- MMLU source manifest: [`data/processed/mmlu_openllm/manifest.json`](data/processed/mmlu_openllm/manifest.json)
- SWE-bench source manifest: [`data/processed/swebench_verified/manifest.json`](data/processed/swebench_verified/manifest.json)
- Data construction: [`scripts/build_m0_data.py`](scripts/build_m0_data.py)

```bash
uv run pytest
uv run ruff check .
uv run mypy src scripts tests
```

---

# What didn't work

### Sequential evaluation was not a universal improvement

It helped SWE-bench at 2–5%, but at 10% its τ was **0.669 versus 0.695** for one-shot evaluation. It was also slightly worse at every tested MMLU budget.

### Better score estimation did not always mean better ordering

The corrected estimator often reduced score MAE, yet the raw stratified mean sometimes produced higher Kendall τ. The current acquisition score is not sufficiently focused on close model pairs.

### The low-rank predictor struggled with later SWE systems

Extra adaptive observations did not consistently repay the audit budget they consumed. Repository, issue-text, difficulty, model-family, and scaffold features are the clearest next improvement.

### This remains a diagnostic study

Ten seeds and normal intervals are enough to select the next design, but not to claim a production fixed-confidence guarantee.